# ExDark → Zero-DCE++ → Bilateral → YOLOv8n

Notebook thực hành trên Kaggle. Chạy từng block từ trên xuống, kiểm tra đầu ra trước khi qua block kế tiếp. Bật **GPU** và **Internet** trong Notebook Settings trước khi bắt đầu.

Zero-DCE++ dùng **official pretrained weights** theo code `src/hienanh`; không huấn luyện lại Zero-DCE++. YOLOv8n được huấn luyện trên ảnh đã qua hai bước xử lý. Các số đo ở cuối phải lấy từ lần chạy thật của bạn.


## Block 1 — Kiểm tra GPU và khai báo thông số chung


In [ ]:
from pathlib import Path
import torch

PROJECT = Path('/kaggle/working/Project18')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'yolov8n.pt'
EPOCHS = 40
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15
SEED = 42
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
assert DEVICE == 'cuda', 'Bật GPU trong Settings → Accelerator rồi chạy lại block này.'


## Block 2 — Lấy project từ GitHub
Nếu clone báo `Could not resolve host`, bật Internet trong Kaggle Settings. Nếu báo không tìm thấy nhánh, hãy push nhánh `feature/denoised-dataset` từ máy của bạn trước.


In [ ]:
import subprocess

REPO_URL = 'https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition.git'
BRANCH = 'feature/denoised-dataset'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT)], check=True)
else:
    print('Project đã tồn tại; dùng bản hiện có:', PROJECT)
assert (PROJECT / 'train_ninh.py').is_file(), 'Thiếu train_ninh.py'
assert (PROJECT / 'src/hienanh/enhancement.py').is_file(), 'Thiếu module Hiền Anh'
assert (PROJECT / 'src/create_denoised_dataset.py').is_file(), 'Thiếu script Bilateral; kiểm tra nhánh đã push.'
subprocess.run(['git', 'branch', '--show-current'], cwd=PROJECT, check=True)


## Block 3 — Cài thư viện và kiểm tra dữ liệu tối


In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
import cv2, yaml, ultralytics
print('OpenCV:', cv2.__version__, '| Ultralytics:', ultralytics.__version__)
DARK = PROJECT / 'Dataset/exdark_yolo_dark'
expected = {'train': 5142, 'valid': 1469, 'test': 734}
for split, count in expected.items():
    images = [p for p in (DARK / split / 'images').iterdir() if p.is_file()]
    labels = list((DARK / split / 'labels').glob('*.txt'))
    print(split, 'images:', len(images), 'labels:', len(labels))
    assert len(images) == len(labels) == count, f'{split}: số ảnh/nhãn không đúng'


## Block 4 — Chuẩn hóa nhãn và kiểm tra rò rỉ giữa các tập
`train` dùng để học; `valid` chọn mô hình; `test` chỉ đánh giá cuối cùng. Đây là bước chuẩn hóa nhãn, chưa train model.


In [ ]:
CLEAN = PROJECT / 'Dataset/exdark_yolo_clean'
if not CLEAN.exists():
    subprocess.run([sys.executable, 'train_ninh.py', 'prepare', '--source', str(DARK), '--output', str(CLEAN), '--seed', str(SEED)], cwd=PROJECT, check=True)
else:
    print('Dataset clean đã tồn tại; kiểm tra trước khi dùng.')
assert (CLEAN / 'data.yaml').is_file()
for split, count in expected.items():
    n_img = len(list((CLEAN / split / 'images').iterdir()))
    n_lab = len(list((CLEAN / split / 'labels').glob('*.txt')))
    print(split, n_img, n_lab)
    assert n_img == n_lab == count


## Block 5 — Tìm hoặc tải official weights Zero-DCE++
`Epoch99.pth` là trọng số do tác giả Zero-DCE++ công bố. Block này kiểm tra SHA-256 (dấu vân tay của file) để tránh dùng nhầm checkpoint.


In [ ]:
import hashlib
WEIGHTS = PROJECT / 'src/hienanh/Results/weights/zerodcepp_Epoch99.pth'
EXPECTED_SHA256 = 'ca8855b90df9a80fa4195a831f33d3476b1964f787eb70602797c773067f3b84'
if not WEIGHTS.is_file():
    print('Chưa có weights trong Kaggle; tải bằng script của Hiền Anh...')
    subprocess.run([sys.executable, 'src/hienanh/download_weights.py'], cwd=PROJECT, check=True)
assert WEIGHTS.is_file(), f'Không tìm thấy {WEIGHTS}'
sha = hashlib.sha256()
with WEIGHTS.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        sha.update(chunk)
assert sha.hexdigest() == EXPECTED_SHA256, 'Checksum sai: checkpoint hỏng hoặc không đúng bản chính thức.'
print('Weights hợp lệ:', WEIGHTS, '| SHA-256:', sha.hexdigest())


## Block 6 — Nạp Zero-DCE++ và thử một ảnh
Scale factor 4 là cấu hình trong phần của Hiền Anh. `training=False` nghĩa là model đang suy luận, không cập nhật trọng số.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(PROJECT))
from src.hienanh.enhancement import load_model, enhance_neural
zero_model = load_model(WEIGHTS, variant='zerodcepp', scale_factor=4, device=DEVICE)
print('Số tham số:', sum(p.numel() for p in zero_model.parameters()), '| training:', zero_model.training)
sample = sorted((CLEAN / 'test/images').iterdir())[0]
raw = cv2.imdecode(np.fromfile(sample, dtype=np.uint8), cv2.IMREAD_COLOR)
assert raw is not None
bright = enhance_neural(raw, zero_model, device=DEVICE)
assert bright.shape == raw.shape
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, image, title in zip(axes, [raw, bright], ['Dark', 'Zero-DCE++']):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); ax.set_title(title); ax.axis('off')
plt.show()
print('Ảnh thử:', sample.name, '| Kích thước:', raw.shape)


## Block 7 — Làm sáng toàn bộ train/valid/test
Block này tạo ảnh PNG cùng kích thước và sao chép nhãn. Nếu Kaggle ngắt phiên, chạy lại block để tiếp tục. Đầu ra có thể cần vài GB dung lượng.


In [ ]:
import json, shutil
ZERO = PROJECT / 'Dataset/exdark_yolo_zerodcepp'
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split, target_count in expected.items():
    in_images, in_labels = CLEAN / split / 'images', CLEAN / split / 'labels'
    out_images, out_labels = ZERO / split / 'images', ZERO / split / 'labels'
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)
    files = sorted(p for p in in_images.iterdir() if p.suffix.lower() in extensions)
    assert len(files) == target_count
    for index, image_path in enumerate(files, 1):
        label = in_labels / (image_path.stem + '.txt')
        dest = out_images / (image_path.stem + '.png')
        out_label = out_labels / label.name
        assert label.is_file(), f'Thiếu nhãn: {label}'
        if not dest.is_file() or not out_label.is_file():
            original = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
            assert original is not None, f'Không đọc được {image_path}'
            result = enhance_neural(original, zero_model, device=DEVICE)
            assert result.shape == original.shape
            ok, encoded = cv2.imencode('.png', result, [cv2.IMWRITE_PNG_COMPRESSION, 3])
            assert ok
            temporary = dest.with_suffix('.png.tmp')
            encoded.tofile(temporary)
            temporary.replace(dest)
            shutil.copy2(label, out_label)
        if index % 250 == 0 or index == len(files):
            print(f'{split}: {index}/{len(files)}', flush=True)
with (CLEAN / 'data.yaml').open(encoding='utf-8-sig') as stream:
    source_config = yaml.safe_load(stream)
zero_config = {'path': str(ZERO.resolve()), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'names': source_config['names']}
(ZERO / 'data.yaml').write_text(yaml.safe_dump(zero_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
metadata = {'method': 'official pretrained Zero-DCE++', 'variant': 'zerodcepp', 'scale_factor': 4, 'checkpoint': str(WEIGHTS), 'checkpoint_sha256': EXPECTED_SHA256, 'source': str(CLEAN), 'train_on_exdark': False, 'counts': expected}
(ZERO / 'enhancement_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
print('Zero-DCE++ dataset:', ZERO)


## Block 8 — Xác minh dataset Zero-DCE++
Kiểm tra tên ảnh và tên nhãn tương ứng, không chỉ đếm tổng số.


In [ ]:
for split, count in expected.items():
    image_stems = {p.stem for p in (ZERO / split / 'images').glob('*.png')}
    label_stems = {p.stem for p in (ZERO / split / 'labels').glob('*.txt')}
    print(split, 'images:', len(image_stems), 'labels:', len(label_stems))
    assert image_stems == label_stems and len(image_stems) == count
print('Dataset Zero-DCE++ đầy đủ.')


## Block 9 — Thử Bilateral trên 5 ảnh mỗi tập
Bilateral Filter (bộ lọc song phương) giảm nhiễu theo khoảng cách pixel và độ giống màu, thường giữ biên tốt hơn lọc mờ thông thường.


In [ ]:
SMOKE = PROJECT / 'Dataset/exdark_yolo_zerodcepp_bilateral_smoke'
if not SMOKE.exists():
    subprocess.run([sys.executable, 'src/create_denoised_dataset.py', '--source', str(ZERO), '--output', str(SMOKE), '--diameter', '5', '--sigma-color', '25', '--sigma-space', '25', '--workers', '4', '--limit', '5'], cwd=PROJECT, check=True)
else:
    print('Smoke dataset đã tồn tại:', SMOKE)
for split in expected:
    assert len(list((SMOKE / split / 'images').iterdir())) == 5
print('Bilateral smoke test đạt.')


## Block 10 — Nhìn ba ảnh cùng nguồn
Quan sát nhiễu, biên vật thể nhỏ, vùng đèn và màu sắc. Nếu lọc làm mờ vật thể, giảm `sigma` hoặc `diameter` trước khi xử lý toàn bộ.


In [ ]:
example = sorted((CLEAN / 'test/images').iterdir())[0]
paths = [example, ZERO / 'test/images' / (example.stem + '.png'), SMOKE / 'test/images' / (example.stem + '.png')]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, path, title in zip(axes, paths, ['Dark', 'Zero-DCE++', 'Zero-DCE++ + Bilateral']):
    image = cv2.imread(str(path))
    assert image is not None, path
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); ax.set_title(title); ax.axis('off')
plt.show()
print('Ảnh so sánh:', example.name)


## Block 11 — Tạo dataset Bilateral đầy đủ
Đây là dữ liệu đưa vào YOLO. Giữ `d=5`, `sigmaColor=25`, `sigmaSpace=25` khi so sánh và ghi đúng các giá trị này trong báo cáo.


In [ ]:
DENOISED = PROJECT / 'Dataset/exdark_yolo_zerodcepp_bilateral_d5_s25'
if not DENOISED.exists():
    subprocess.run([sys.executable, 'src/create_denoised_dataset.py', '--source', str(ZERO), '--output', str(DENOISED), '--diameter', '5', '--sigma-color', '25', '--sigma-space', '25', '--workers', '4'], cwd=PROJECT, check=True)
else:
    print('Dataset Bilateral đã tồn tại; kiểm tra tính đầy đủ.')
assert (DENOISED / 'data.yaml').is_file()
for split, count in expected.items():
    image_stems = {p.stem for p in (DENOISED / split / 'images').iterdir() if p.is_file()}
    label_stems = {p.stem for p in (DENOISED / split / 'labels').glob('*.txt')}
    print(split, len(image_stems), len(label_stems))
    assert image_stems == label_stems and len(image_stems) == count
print('Dataset sẵn sàng cho YOLO:', DENOISED / 'data.yaml')


## Block 12 — Train thử YOLOv8n trong 2 epoch
Smoke test giúp kiểm tra data.yaml, nhãn, GPU và bộ nhớ. Nếu thiếu VRAM, đổi `BATCH_SIZE = 8` ở Block 1 rồi chạy lại từ Block 12.


In [ ]:
RESULTS = PROJECT / 'Results/tiep'
smoke_run = RESULTS / 'yolo_zerodcepp_bilateral_smoke'
assert MODEL_NAME == 'yolov8n.pt', 'train_ninh.py hiện cố định model yolov8n.pt'
if not smoke_run.exists():
    subprocess.run([sys.executable, 'train_ninh.py', 'train', '--data', str(DENOISED / 'data.yaml'), '--project', str(RESULTS), '--name', smoke_run.name, '--epochs', '2', '--batch', str(BATCH_SIZE), '--imgsz', str(IMAGE_SIZE), '--device', '0', '--hsv-v', '0.1', '--close-mosaic', '2', '--patience', '2', '--seed', str(SEED)], cwd=PROJECT, check=True)
else:
    print('Run smoke đã tồn tại:', smoke_run)
assert (smoke_run / 'weights/best.pt').is_file(), 'Smoke test chưa hoàn thành; kiểm tra log và dùng tên run mới nếu cần.'


## Block 13 — Huấn luyện YOLOv8n chính thức
Thông số: 40 epoch tối đa, ảnh 640, batch 16, patience 15, seed 42. `best.pt` là checkpoint tốt nhất theo validation.


In [ ]:
YOLO_RUN = RESULTS / 'yolo_zerodcepp_bilateral_e40'
if not YOLO_RUN.exists():
    subprocess.run([sys.executable, 'train_ninh.py', 'train', '--data', str(DENOISED / 'data.yaml'), '--project', str(RESULTS), '--name', YOLO_RUN.name, '--epochs', str(EPOCHS), '--batch', str(BATCH_SIZE), '--imgsz', str(IMAGE_SIZE), '--device', '0', '--hsv-v', '0.1', '--close-mosaic', '10', '--patience', str(PATIENCE), '--seed', str(SEED)], cwd=PROJECT, check=True)
else:
    print('Run YOLO đã tồn tại:', YOLO_RUN)
BEST = YOLO_RUN / 'weights/best.pt'
assert BEST.is_file(), 'YOLO run chưa có best.pt; kiểm tra log. Script hiện không tự resume.'
print('Checkpoint YOLO:', BEST)


## Block 14 — Kiểm tra tham số YOLO đã dùng


In [ ]:
args = yaml.safe_load((YOLO_RUN / 'args.yaml').read_text())
for key in ['model', 'epochs', 'imgsz', 'batch', 'patience', 'seed', 'hsv_v', 'close_mosaic']:
    print(key, ':', args.get(key))
assert args['epochs'] == EPOCHS and args['imgsz'] == IMAGE_SIZE and args['batch'] == BATCH_SIZE
assert args['patience'] == PATIENCE and args['seed'] == SEED


## Block 15 — Đánh giá trên test đúng một lần
Precision là tỷ lệ dự đoán đúng trong các dự đoán; recall là tỷ lệ vật thể thật được tìm thấy; mAP50-95 đánh giá hộp ở nhiều ngưỡng khớp và khắt khe hơn mAP50.


In [ ]:
TEST_RUN = RESULTS / 'test_zerodcepp_bilateral'
metrics_file = TEST_RUN / 'measured_metrics.json'
if not metrics_file.is_file():
    assert not TEST_RUN.exists(), 'Thư mục đánh giá dở dang đã tồn tại; dùng tên run mới sau khi kiểm tra.'
    subprocess.run([sys.executable, 'train_ninh.py', 'evaluate', '--data', str(DENOISED / 'data.yaml'), '--weights', str(BEST), '--project', str(RESULTS), '--name', TEST_RUN.name, '--split', 'test', '--batch', str(BATCH_SIZE), '--imgsz', str(IMAGE_SIZE), '--device', '0'], cwd=PROJECT, check=True)
metrics = json.loads(metrics_file.read_text())
print(json.dumps(metrics, indent=2))


## Block 16 — Xuất bảng số liệu thật
Bảng này chỉ có một hàng của thí nghiệm bạn làm. Ghép với Raw và Zero-DCE++ của nhóm để kiểm tra Bilateral có giúp mAP không.


In [ ]:
import pandas as pd
table = pd.DataFrame([{'experiment': 'Zero-DCE++ + Bilateral + YOLOv8n', 'enhancer': 'official pretrained Zero-DCE++', 'bilateral_d': 5, 'sigma_color': 25, 'sigma_space': 25, 'epochs_max': EPOCHS, 'imgsz': IMAGE_SIZE, 'batch': BATCH_SIZE, 'patience': PATIENCE, 'seed': SEED, 'precision': metrics['precision'], 'recall': metrics['recall'], 'map50': metrics['map50'], 'map50_95': metrics['map50_95']}])
summary_file = RESULTS / 'dcepp_bilateral_summary.csv'
table.to_csv(summary_file, index=False)
display(table)
print('Bảng:', summary_file)


## Block 17 — Xem biểu đồ và ma trận nhầm lẫn


In [ ]:
from IPython.display import Image, display
for path in [YOLO_RUN / 'results.png', TEST_RUN / 'confusion_matrix.png', TEST_RUN / 'PR_curve.png', TEST_RUN / 'F1_curve.png']:
    print(path.name, '→', path.exists())
    if path.exists():
        display(Image(filename=str(path)))


## Block 18 — Xuất ảnh YOLO dự đoán
`conf=0.25` chỉ ảnh hưởng ảnh minh họa được vẽ, không thay đổi mAP đã tính trong Block 15.


In [ ]:
PRED_RUN = RESULTS / 'pred_zerodcepp_bilateral'
if not PRED_RUN.exists():
    subprocess.run(['yolo', 'detect', 'predict', f'model={BEST}', f'source={DENOISED / "test/images"}', f'imgsz={IMAGE_SIZE}', 'conf=0.25', 'device=0', 'save=True', 'save_txt=True', 'save_conf=True', f'project={RESULTS}', f'name={PRED_RUN.name}'], cwd=PROJECT, check=True)
images = sorted(p for p in PRED_RUN.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print('Số ảnh dự đoán:', len(images))
assert images, 'Chưa có ảnh dự đoán'
for image_path in images[:6]:
    display(Image(filename=str(image_path)))


## Block 19 — Kiểm tra đầu ra và tải kết quả
File ZIP chứa checkpoint, số liệu, biểu đồ và ảnh dự đoán. Dataset ảnh xử lý nằm riêng trong `/kaggle/working/Project18/Dataset` và không được đưa vào ZIP này để tránh file quá lớn. Lưu phiên bản Kaggle hoặc tải ZIP trước khi kết thúc session.


In [ ]:
from IPython.display import FileLink
required = [WEIGHTS, ZERO / 'enhancement_metadata.json', DENOISED / 'denoising_metadata.json', DENOISED / 'data.yaml', BEST, YOLO_RUN / 'args.yaml', YOLO_RUN / 'results.csv', metrics_file, summary_file]
for path in required:
    print('OK' if path.is_file() else 'THIẾU', path)
assert all(path.is_file() for path in required)
archive = shutil.make_archive('/kaggle/working/tiep_dcepp_bilateral_results', 'zip', root_dir=RESULTS)
print('Đã tạo:', archive)
display(FileLink(archive))


## Câu hỏi khi viết báo cáo
1. Ảnh qua Zero-DCE++ có sáng hơn nhưng nhiễu có tăng không?
2. Bilateral giảm nhiễu có làm mất biên hoặc vật thể nhỏ không?
3. So với **YOLOv8n trên Raw Dark** và **YOLOv8n trên Zero-DCE++** (cùng split, seed, epochs, imgsz), mAP50-95 tăng hay giảm?
4. Đưa cả ảnh thành công và ảnh thất bại vào báo cáo. Không dùng số liệu minh họa viết sẵn trong notebook khác.
